# S&P 500 / Nasdaq-100: Covariance, PCA & Random Matrix Theory

One-click analysis (no local setup). Inspired by **Laloux et al. (1999)** — *Random Matrix Theory and Financial Correlations*.

**Step 1** — Sample covariance & correlation matrices from daily returns (since 2020)  
**Step 2** — PCA eigendecomposition + Marchenko-Pastur eigenvalue spectrum

> Runtime: ~2–4 minutes (mostly downloading price data from Yahoo Finance).

In [ ]:
# Install dependencies (Colab has most of these; pyarrow + yfinance are the key adds)
!pip install -q numpy pandas scipy yfinance matplotlib seaborn requests lxml html5lib pyarrow

In [ ]:
# Clone the analysis repo and run the pipeline
import os
from pathlib import Path

REPO_URL = "https://github.com/marka789/for_fun.git"
BRANCH = "cursor/sp500-pca-rmt-analysis-d3bf"  # switch to "main" after PR merge

if not Path("for_fun/run_analysis.py").exists():
    !git clone -b {BRANCH} --depth 1 {REPO_URL} for_fun

%cd for_fun
!python run_analysis.py

In [ ]:
# Summary table
import json
import pandas as pd
from IPython.display import display

with open("outputs/analysis_summary.json") as f:
    summary = json.load(f)

rows = []
for name, r in summary["universes"].items():
    rows.append({
        "Universe": name.upper(),
        "Stocks (N)": r["n_assets"],
        "Days (T)": r["n_observations"],
        "q = N/T": round(r["q_ratio"], 4),
        "Market λ₁": round(r["top_eigenvalue"], 1),
        "Market % var": f"{100*r['top_pc_variance_share']:.1f}%",
        "Signal PCs": r["n_signal_eigenvalues"],
        "Signal % var": f"{100*r['variance_explained_signal']:.1f}%",
        "Noise PCs": r["n_noise_eigenvalues"],
    })

display(pd.DataFrame(rows))

In [ ]:
# S&P 500 — eigenvalue spectrum vs Marchenko-Pastur
from IPython.display import Image, display

for universe in ["sp500", "nasdaq100"]:
    print(f"\n{'='*50}\n  {universe.upper()}\n{'='*50}")
    display(Image(filename=f"outputs/{universe}_eigenvalue_spectrum.png"))
    display(Image(filename=f"outputs/{universe}_pca_scree.png"))

In [ ]:
# Top principal component loadings
from IPython.display import Image, display

display(Image(filename="outputs/sp500_pca_loadings.png"))

## Download outputs

Run the cell below to zip all matrices, summaries, and plots to your Google Drive / local downloads.

In [ ]:
!zip -qr analysis_outputs.zip outputs/
from google.colab import files
files.download("analysis_outputs.zip")